# CS 123 — Teach Your Pupper a Trick 🐾

Design a motion, check it with physics, train a small policy that tracks it
(BeyondMimic-style), and play it on the robot with a voice command.

The loop:

1. **Design** the motion as keyframes (poses in metres and degrees), solved with IK.
2. **Replay** it with physics. If plain replay holds up, you're done: the robot can play the recording.
3. **Upload** the motion to Weights & Biases. From here on everything moves by name, not by file.
4. **Train** a tracking policy on it, about 15 minutes. The policy lands on your W&B run automatically.
5. **Fetch** it on the robot with one command, and ask Gemini Pupper to do it.

*Runtime → Change runtime type → a GPU runtime (G4 recommended), then run cells top to bottom.*

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/cs123-stanford/pupper-mjlab.git"
BRANCH = "main"

import os

!nvidia-smi -L
if not os.path.exists("/content/pupper-mjlab"):
  !git clone --branch {BRANCH} --depth 1 {REPO_URL} /content/pupper-mjlab
%cd /content/pupper-mjlab
!pip install -q -e .

# The running kernel can't see an editable install's path hooks; wire src/
# onto the path so imports work without a runtime restart.
import sys

sys.path.insert(0, "/content/pupper-mjlab/src")
for _m in [k for k in list(sys.modules) if k == "mjlab" or k.startswith("mjlab.")]:
  del sys.modules[_m]
import mjlab.envs  # smoke-test

print("mjlab ready")

## 2. Log into Weights & Biases

W&B carries your motion from this notebook to training, and your trained policy
from training to the robot. **Use the same account your Pupper is logged into**
(the one you deploy walking policies with).

In [ ]:
WANDB_API_KEY = ""  # paste your key from https://wandb.ai/authorize between the quotes

import wandb

if WANDB_API_KEY:
  os.environ["WANDB_API_KEY"] = WANDB_API_KEY
  wandb.login(key=WANDB_API_KEY, relogin=True)
else:
  wandb.login()  # no key pasted: falls back to the interactive prompt

## 3. Design the motion ✏️

A motion is a list of **keyframes**: at time `t` (seconds), where the body is
(`root_pos`, metres), how it is tilted (`root_rpy`, degrees: roll, pitch, yaw;
negative pitch = nose up), and where each foot is (`feet`, world metres).
`build_motion` interpolates between keyframes at 50 Hz and solves inverse
kinematics for the joint angles.

The cell starts from the worked example, the **sit** (`tricks/sit.py`: read its
comments). Change it into your trick. A coding agent is a good partner here:
describe the pose in words and numbers, and let it search for keyframe values.

Rules of thumb:
- Start and end standing (`md.STAND_HEIGHT`, `md.stand_feet()`), unless you design a clip trick (section 3b).
- Keep the centre of mass over the feet on the ground; lean away from a foot before lifting it.
- Move slowly at first. Real motors have speed limits; the report below flags fast joints.

In [ ]:
from pathlib import Path

from mjlab.tasks.tracking.config.pupper import motion_design as md
from mjlab.tasks.tracking.config.pupper.tricks import sit

TRICK_NAME = "my_trick"  # lowercase, no spaces: the name on W&B and on the robot
OUT = Path("/content/tricks")
OUT.mkdir(exist_ok=True)


def my_keyframes() -> list[md.Keyframe]:
  # TODO: your trick. The worked example, unchanged:
  return sit.keyframes()


motion = md.build_motion(my_keyframes(), fps=50.0)
for key, value in motion.report.items():
  print(f"{key:36} {value}")
MOTION_FILE = motion.save_npz(OUT / f"{TRICK_NAME}.npz")  # for training
motion.save_robot_csv(OUT / f"{TRICK_NAME}_robot.csv")  # for plain replay on the robot
print("saved", MOTION_FILE)

Read the report before going on: IK error should stay under a few millimetres,
nothing should go below the ground, and no joint should sit at its limit.

### 3b. Optional: a clip trick

Some tricks are a pose the robot *holds* plus moves it can do from that pose on
request. Design each piece as its own motion and join them into a **clip
library**: one policy trains on all of it, and on the robot the controller plays
one clip at a time (see "Clip tricks" in
`src/mjlab/tasks/tracking/config/pupper/README.md`). Clips must meet at matching
poses. Skip this cell for a single motion.

In [ ]:
USE_CLIPS = False

if USE_CLIPS:
  # TODO: build each clip with md.build_motion, then:
  clips = [
    # md.Clip("get_into_pose", intro_motion, next="hold"),
    # md.Clip("hold", hold_motion, loop=True),
    # md.Clip("move_a", move_a_motion, next="hold"),
    # md.Clip("leave", leave_motion),
  ]
  motion = md.concat_clips(clips)  # pass order=[...] to train a longer sequence
  for c in motion.clips:
    print(c)
  MOTION_FILE = motion.save_npz(OUT / f"{TRICK_NAME}.npz")

## 4. Replay it with physics

Plays the joint trajectory on the simulated robot with the robot's
trick-player gains (no policy) and prints the tracking error and a verdict:

- **HOLDS UP**: plain replay works. Train anyway: on the robot every trick runs as a policy, and a policy also catches slips and pushes.
- **WOBBLES**: it stays up but drifts. A policy will be tighter.
- **FALLS**: plain replay won't work. Make the motion gentler, or train it.

In [ ]:
!python -m mjlab.tasks.tracking.config.pupper.play_motion {MOTION_FILE}

To *see* it, open the viewer below (a public link). The translucent ghost is
the reference; stop the cell when you're done. Add `--no-physics` to watch the
reference alone.

In [ ]:
!python -m mjlab.tasks.tracking.config.pupper.play_motion {MOTION_FILE} --viewer viser --share True --loops 3

## 5. Upload the motion to W&B

One upload, then everything refers to it by name
(`<your-entity>/mjlab/<TRICK_NAME>`): training pulls it, and the robot can
fetch its recording for plain replay. Re-uploading after a change makes a new
version; training uses the latest.

In [ ]:
!python -m mjlab.tasks.tracking.config.pupper.upload_motion {MOTION_FILE} --name {TRICK_NAME}

MOTION_ARTIFACT = f"{wandb.Api().default_entity}/mjlab/{TRICK_NAME}"
print("motion artifact:", MOTION_ARTIFACT)

On the robot, every trick runs as a policy (a recording can't catch
itself when it slips), so the next step is training, even if plain replay
held up.

## 6. Train 🚀

About 15 minutes for 3,000 iterations. Open the printed W&B link to watch.
The numbers that matter are `Metrics/motion/error_*` (lower is better) and
`Episode_Reward/motion_*` (close to 1 is good).

The trick policy is uploaded to the run as `policy.json` at every checkpoint
and when you press stop, and the log prints the exact command for the robot.

In [ ]:
from mjlab.scripts.train import TrainConfig, launch_training
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg
import mjlab.tasks  # register tasks

TASK = "Mjlab-Tracking-Flat-Pupper-v3"
ITERATIONS = 3000

env_cfg = load_env_cfg(TASK)
env_cfg.scene.num_envs = 4096
agent_cfg = load_rl_cfg(TASK)
agent_cfg.max_iterations = ITERATIONS
agent_cfg.logger = "wandb"
agent_cfg.run_name = TRICK_NAME

launch_training(TASK, TrainConfig(env=env_cfg, agent=agent_cfg, registry_name=MOTION_ARTIFACT))

## 7. Score it

The same physics check as section 4, now with your policy in charge. Compare
the verdict and the worst errors with plain replay: that is what training
bought you.

In [ ]:
runs = sorted(Path("logs/rsl_rl/pupper_tracking").glob("*"), key=lambda p: p.stat().st_mtime)
assert runs, "No runs yet -- train first."
CHECKPOINT = max(runs[-1].glob("model_*.pt"), key=lambda p: int(p.stem.split("_")[1]))
print("checkpoint:", CHECKPOINT)
!python -m mjlab.tasks.tracking.config.pupper.play_motion {MOTION_FILE} --checkpoint {CHECKPOINT}

Watch the policy (public link; stop the cell when done):

In [ ]:
!python -m mjlab.tasks.tracking.config.pupper.play_motion {MOTION_FILE} --checkpoint {CHECKPOINT} --viewer viser --share True --loops 3

## 8. On the robot 🤖

In your gemini_pupper repo on the Pupper (logged into the same W&B account),
fetch the policy with the command the training log printed:

```bash
python3 robot/get_trick.py <entity>/mjlab/<run-id> --name <TRICK_NAME>
```

It downloads and validates the policy into `tricks/<TRICK_NAME>/` and writes a
`trick.yaml` template. **Edit its `description`**: it is what Gemini reads
when deciding whether your trick matches what someone asked for. For a clip
trick, also fill in `actions` and `exit_clip`. Then restart Gemini Pupper
(`robot/stop.sh`, then `robot/start.sh`) and ask for your trick.

Safety: test new tricks with the robot on the ground, clear space around it,
and a hand near the e-stop. When you're done here: **Runtime → Disconnect**.